**Description** : Transformation du graphe en un problème de Machine Learning supervisé.

**Objectifs** : Création d'un dataset équilibré incluant des liens réels et des liens fictifs (Negative Sampling).

**Feature Engineering** : Extraction de caractéristiques topologiques (degrés, attachement préférentiel), communautaires (ID de cluster) et attributaires.

**Cellule 1 : Chargement des données et des résultats précédents**

In [1]:
#Importation les bibliothéques nécessaires
import pandas as pd
import networkx as nx
import numpy as np
import random

In [2]:
# 1. Données originales et Clusters
df = pd.read_csv("../data/linkedin_cleaned_data.csv")
df_clusters = pd.read_csv("../resultats/affectation_clusters_louvain.csv")

Ici, j'ai chargé les 2 datasets. Les données qu'on a nettoyé dans le 1ére notebook et l'autre qui inclut les affectation de nos employés dans différents clusters en utilisant les résultats de l'algorithme "Louvain".

In [3]:
# 2. On recrée le graphe biparti (Employé - Entreprise)
# On suppose que 'current_company_name' est l'entreprise cible
B = nx.Graph()
B.add_nodes_from(df['id'], bipartite=0) # Employés
B.add_nodes_from(df['current_company_name'].unique(), bipartite=1) # Entreprises

edges = [(row['id'], row['current_company_name']) for _, row in df.iterrows()]
B.add_edges_from(edges)

print(f"✅ Graphe biparti prêt : {B.number_of_nodes()} nœuds, {B.number_of_edges()} liens existants.")

✅ Graphe biparti prêt : 1803 nœuds, 1000 liens existants.


Juste ici on a retourné à notre graphe biparti pour modéliser les liens entre nos employés et les entreprises

**Cellule 2 : Échantillonnage Négatif (Negative Sampling)**

Pour apprendre, le modèle doit voir des cas où l'employé n'est pas dans l'entreprise.

In [5]:
# On récupère tous les employés et toutes les entreprises
all_employees = df['id'].tolist()
all_companies = df['current_company_name'].unique().tolist()

In [6]:
# Liste des liens positifs (Label 1)
positive_samples = edges
labels_pos = [1] * len(positive_samples)

In [7]:
# Génération de liens négatifs (Label 0)
negative_samples = []
while len(negative_samples) < len(positive_samples):
    emp = random.choice(all_employees)
    comp = random.choice(all_companies)
    
    # On vérifie que le lien n'existe pas déjà
    if not B.has_edge(emp, comp):
        negative_samples.append((emp, comp))

labels_neg = [0] * len(negative_samples)

## 🔷 Génération des exemples négatifs (Label = 0)

Dans le cadre de la prédiction de liens, il est nécessaire de construire un jeu de données contenant :

* des exemples **positifs (1)** : liens existants dans le graphe (employé → entreprise)
* des exemples **négatifs (0)** : liens inexistants

---

### 🔹 Principe

Les exemples négatifs sont générés de manière aléatoire en sélectionnant :

* un employé
* une entreprise

puis en vérifiant que le lien entre eux **n’existe pas déjà** dans le graphe.

---

### 🔹 Fonctionnement du code

1. Initialisation d’une liste vide pour stocker les exemples négatifs
2. Génération aléatoire de paires (employé, entreprise)
3. Vérification que le lien n’existe pas dans le graphe
4. Ajout de la paire comme exemple négatif
5. Répétition jusqu’à obtenir le même nombre d’exemples négatifs que positifs
6. Attribution du label 0 à ces exemples

---

### 🔹 Objectif

L’objectif est de créer un dataset équilibré, contenant autant de cas positifs que négatifs, afin d’entraîner efficacement un modèle de prédiction de liens.

---

### 🔹 Remarque

Les exemples négatifs ne signifient pas que le lien est impossible, mais simplement qu’il n’est **pas observé dans les données**.

---

### 🔹 Conclusion

Cette étape est essentielle pour transformer le problème de prédiction de liens en un problème de classification supervisée.


In [8]:
# Fusion des deux listes
all_samples = positive_samples + negative_samples
all_labels = labels_pos + labels_neg

In [9]:
df_ml = pd.DataFrame(all_samples, columns=['employee_id', 'company_name'])
df_ml['target'] = all_labels

print(f"✅ Dataset créé : {len(df_ml)} exemples (50% positifs / 50% négatifs).")

✅ Dataset créé : 2000 exemples (50% positifs / 50% négatifs).


**Cellule 3 : Création des Features (Topologiques, Communauté, Attributs)**

L'objectif de cette étape d'enrichir  nos donnée d'entrainement en ajoutant d'autres attributs tels que des attributs toplogiques basés sur des mesures comme les degrés et preferential attachement , des attributs basé sur les commauntés et d'autres basé sur l'industrie et company

In [12]:
# Préparation des dictionnaires pour un accès rapide
cluster_map = dict(zip(df_clusters['employee_id'].astype(str), df_clusters['cluster_id']))
emp_info = df.set_index('id').to_dict('index')

# On calcule le degré pour chaque nœud (Feature Topologique)
degres = dict(B.degree())

features = []

for _, row in df_ml.iterrows():
    emp_id = row['employee_id']
    comp_name = row['company_name']
    
    # --- 1. Features Topologiques ---
    # Nombre d'entreprises connues de l'employé et taille de l'entreprise
    emp_deg = degres.get(emp_id, 0)
    comp_deg = degres.get(comp_name, 0)
    pref_attachment = emp_deg * comp_deg
    
    # --- 2. Features de Communauté ---
    # On récupère le cluster de l'employé
    emp_cluster = cluster_map.get(str(emp_id), -1)
    
    # --- 3. Features Attributaires (Similarité) ---
    # On va comparer l'employé avec le profil "type" de l'entreprise
    # (Par exemple, est-ce que l'employé est dans la même industrie que l'entreprise ?)
    info = emp_info.get(emp_id, {})
    
    # On cherche si l'employé partage des points communs avec l'entreprise cible 
    # (via les autres employés de cette entreprise)
    comp_employees = [n for n in B.neighbors(comp_name)]
    
    same_location = 0
    same_industry = 0
    
    if comp_employees:
        # On regarde si l'employé est dans la même ville/industrie que la majorité de l'entreprise
        for ce_id in comp_employees[:10]: # On regarde un échantillon pour aller vite
            ce_info = emp_info.get(ce_id, {})
            if info.get('city') == ce_info.get('city'): same_location = 1
            if info.get('industry') == ce_info.get('industry'): same_industry = 1

    features.append({
        'emp_degree': emp_deg,
        'comp_degree': comp_deg,
        'preferential_attachment': pref_attachment,
        'cluster_id': emp_cluster,
        'same_location': same_location,
        'same_industry': same_industry
    })

# Ajout des features au DataFrame
df_features = pd.DataFrame(features)
df_final_ml = pd.concat([df_ml, df_features], axis=1)

print("✅ Features extraites avec succès.")

✅ Features extraites avec succès.


## 🔷 Extraction des features pour la prédiction de liens

Dans cette étape, nous construisons un ensemble de caractéristiques (features) pour chaque paire employé–entreprise afin d’entraîner un modèle de Machine Learning pour la prédiction de liens.

---

### 🔹 Objectif

Transformer le graphe en un problème de classification supervisée en construisant des variables explicatives pour chaque couple (employé, entreprise).

---

### 🔹 Types de features utilisées

#### ✔️ 1. Features topologiques

Ces features sont basées sur la structure du graphe :

* degré de l’employé
* degré de l’entreprise
* preferential attachment (produit des degrés)

---

#### ✔️ 2. Features de communauté

Chaque employé est associé à une communauté détectée précédemment (Louvain).
Cette information permet de capturer la structure globale du réseau.

---

#### ✔️ 3. Features basées sur les attributs

On compare les caractéristiques de l’employé avec celles des employés déjà présents dans l’entreprise :

* même ville
* même industrie

---

### 🔹 Construction du dataset

Toutes les features sont regroupées dans un seul DataFrame final, qui sera utilisé pour l’entraînement des modèles de Machine Learning.

---

### 🔹 Conclusion

Cette étape permet de combiner :

* la structure du graphe
* les communautés
* les attributs des individus

afin d’améliorer la performance de la prédiction de liens.


## 🔷 Preferential Attachment (Attachement préférentiel)

Le **preferential attachment** est une feature topologique utilisée en analyse de graphes pour estimer la probabilité qu’un lien existe entre deux nœuds.





---

### 🔹 Idée principale

Le principe est simple :

👉 plus un nœud est connecté (important dans le réseau),
👉 plus il a de chances de créer de nouvelles connexions.

---

### 🔹 Interprétation dans notre cas

Dans un graphe employé–entreprise :

* un employé très connecté (profil similaire à beaucoup d’entreprises) a plus de chances d’être relié à d’autres entreprises
* une entreprise avec beaucoup d’employés est plus susceptible d’attirer de nouveaux employés

---

### 🔹 Utilité

Le preferential attachment permet de capturer un phénomène naturel des réseaux :

👉 les nœuds populaires ont tendance à attirer encore plus de connexions.

---

### 🔹 Conclusion

Cette feature permet d’intégrer l’effet de popularité dans le modèle de prédiction de liens et améliore la capacité à identifier les connexions probables dans le graphe.


**Cellule 4 : Sauvegarde du Dataset final**

In [13]:
chemin_ml = "../resultats/dataset_prediction_liens.csv"
df_final_ml.to_csv(chemin_ml, index=False)

print(f"💾 Dataset prêt pour le Machine Learning sauvegardé dans : {chemin_ml}")

💾 Dataset prêt pour le Machine Learning sauvegardé dans : ../resultats/dataset_prediction_liens.csv
